# ⚡ Notebook 5: Application Caching with Redis

Caching is the most effective way to scale reads. Redis sits between your application and database, serving frequently accessed data from memory.

## Learning Objectives

By the end of this notebook, you'll understand:
- How application-level caching works
- Cache-aside pattern
- TTL strategies
- Basic cache invalidation

---

### 🔍 Open RedisInsight to Watch Cache Operations!

1. Go to **http://localhost:5540**
2. Click "Add Redis Database" → Host: `redis`, Port: `6379`
3. Open the **Browser** tab to see keys as they're created!

In [ ]:
import redis
import psycopg2
import json
import time
from typing import Optional

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "scaling_demo",
    "user": "demo",
    "password": "demo"
}

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

try:
    r.ping()
    print("✅ Redis connected")
except:
    print("❌ Redis not running. Start with: docker-compose up -d")

try:
    conn = get_db_connection()
    print("✅ PostgreSQL connected")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

## 🏗️ Cache-Aside Pattern

The most common caching pattern: check cache first, fall back to database.

In [ ]:
print("🏗️ Cache-Aside Pattern")
print("=" * 60)
print("""
READ PATH:
─────────────────────────────────────────────────────────────

    Application
        │
        ├──► 1. Check Cache
        │         │
        │    ┌────┴────┐
        │    │         │
        │  HIT ✅    MISS ❌
        │    │         │
        │    │    2. Query Database
        │    │         │
        │    │    3. Store in Cache
        │    │         │
        └────┴─────────┘
                │
           Return Data

─────────────────────────────────────────────────────────────

• Cache HIT:  ~1ms  (memory read)
• Cache MISS: ~10ms (DB query + cache write)
""")

In [ ]:
class CachedUserRepository:
    def __init__(self, redis_client, ttl_seconds: int = 300):
        self.redis = redis_client
        self.ttl = ttl_seconds
        self.stats = {"hits": 0, "misses": 0}
    
    def _cache_key(self, user_id: int) -> str:
        return f"user:{user_id}"
    
    def get_user(self, user_id: int) -> Optional[dict]:
        cache_key = self._cache_key(user_id)
        
        cached = self.redis.get(cache_key)
        if cached:
            self.stats["hits"] += 1
            return json.loads(cached)
        
        self.stats["misses"] += 1
        
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute(
            "SELECT id, username, email, display_name, follower_count "
            "FROM users WHERE id = %s", (user_id,)
        )
        row = cursor.fetchone()
        conn.close()
        
        if not row:
            return None
        
        user = {
            "id": row[0],
            "username": row[1],
            "email": row[2],
            "display_name": row[3],
            "follower_count": row[4]
        }
        
        self.redis.setex(cache_key, self.ttl, json.dumps(user))
        
        return user
    
    def invalidate(self, user_id: int):
        self.redis.delete(self._cache_key(user_id))

repo = CachedUserRepository(r, ttl_seconds=60)
print("✅ CachedUserRepository created with 60s TTL")

In [ ]:
print("🔬 Testing Cache-Aside Pattern")
print("=" * 60)

repo.invalidate(1)

print("\n📖 First read (cache MISS):")
start = time.time()
user = repo.get_user(1)
elapsed = (time.time() - start) * 1000
print(f"   Time: {elapsed:.2f}ms")
print(f"   User: {user['username']}")
print(f"   Stats: {repo.stats}")

print("\n📖 Second read (cache HIT):")
start = time.time()
user = repo.get_user(1)
elapsed = (time.time() - start) * 1000
print(f"   Time: {elapsed:.2f}ms")
print(f"   User: {user['username']}")
print(f"   Stats: {repo.stats}")

print("\n💡 Second read was MUCH faster - came from cache!")
print("   👀 Check RedisInsight for 'user:1' key!")

## ⏰ TTL Strategies

In [ ]:
print("⏰ TTL (Time-To-Live) Strategies")
print("=" * 60)
print("""
TTL determines how long cached data lives before expiring.

SHORT TTL (5-60 seconds)
─────────────────────────────────────────────────────────────
• Use for: Frequently changing data
• Examples: Stock prices, live scores, inventory counts
• Trade-off: More cache misses, but fresher data

MEDIUM TTL (5-15 minutes)
─────────────────────────────────────────────────────────────
• Use for: Semi-static data
• Examples: User profiles, product details, blog posts
• Trade-off: Good balance of freshness and hit rate

LONG TTL (1 hour - 24 hours)
─────────────────────────────────────────────────────────────
• Use for: Rarely changing data
• Examples: Config settings, category lists, translations
• Trade-off: High hit rate, but may serve stale data

NO TTL (Never expires)
─────────────────────────────────────────────────────────────
• Use for: Immutable data
• Examples: URL shortener mappings, historical data
• Trade-off: Must invalidate manually on changes
""")

In [ ]:
print("🔬 Demonstrating TTL Expiration")
print("=" * 60)

r.setex("demo:short_ttl", 3, "I expire in 3 seconds")
print("\n✏️ Set key with 3 second TTL")

print("\n⏱️ Watching TTL countdown:")
for i in range(5):
    value = r.get("demo:short_ttl")
    ttl = r.ttl("demo:short_ttl")
    status = "✅" if value else "❌ EXPIRED"
    print(f"   Second {i}: TTL={ttl}s - {status}")
    time.sleep(1)

print("\n💡 After TTL expires, next read causes cache miss!")

## 🔄 Cache Invalidation Strategies

In [ ]:
print("🔄 Cache Invalidation Strategies")
print("=" * 60)
print("""
1. TIME-BASED (TTL)
─────────────────────────────────────────────────────────────
• Simplest approach - data expires after fixed time
• Pro: No extra code needed
• Con: May serve stale data until TTL expires

2. WRITE-THROUGH
─────────────────────────────────────────────────────────────
• Update cache immediately when writing to DB
• Pro: Cache always has latest data
• Con: Adds latency to writes

3. WRITE-BEHIND (INVALIDATE)
─────────────────────────────────────────────────────────────
• Delete cache entry on write, let next read populate
• Pro: Simple, eventual consistency
• Con: Next read has cache miss

4. EVENT-DRIVEN
─────────────────────────────────────────────────────────────
• Publish event on write, subscriber invalidates
• Pro: Decoupled, works across services
• Con: More complex infrastructure
""")

In [ ]:
class CachedUserRepositoryWithInvalidation(CachedUserRepository):
    def update_user(self, user_id: int, display_name: str):
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute(
            "UPDATE users SET display_name = %s WHERE id = %s",
            (display_name, user_id)
        )
        conn.commit()
        conn.close()
        
        self.invalidate(user_id)
        print(f"   🗑️ Invalidated cache for user:{user_id}")

repo2 = CachedUserRepositoryWithInvalidation(r, ttl_seconds=300)

print("🔬 Write-Behind Invalidation Demo")
print("=" * 60)

print("\n1. Read user (populates cache):")
user = repo2.get_user(1)
print(f"   Display name: {user['display_name']}")

print("\n2. Update user (invalidates cache):")
repo2.update_user(1, "New Display Name")

print("\n3. Read again (cache miss, gets fresh data):")
user = repo2.get_user(1)
print(f"   Display name: {user['display_name']}")
print(f"   Stats: {repo2.stats}")

## 📊 Cache Hit Rate

In [ ]:
import random

def simulate_traffic(repo, num_requests: int, popular_users: list, all_users: range):
    for user_id in popular_users:
        repo.invalidate(user_id)
    repo.stats = {"hits": 0, "misses": 0}
    
    for _ in range(num_requests):
        if random.random() < 0.8:
            user_id = random.choice(popular_users)
        else:
            user_id = random.choice(list(all_users))
        repo.get_user(user_id)
    
    total = repo.stats["hits"] + repo.stats["misses"]
    hit_rate = (repo.stats["hits"] / total) * 100 if total > 0 else 0
    return hit_rate, repo.stats

print("📊 Cache Hit Rate Analysis")
print("=" * 60)

repo3 = CachedUserRepository(r, ttl_seconds=60)

popular = [1, 2, 3, 4, 5]
all_users = range(1, 101)

hit_rate, stats = simulate_traffic(repo3, 1000, popular, all_users)

print(f"\n📊 Results after 1000 requests:")
print(f"   Hits: {stats['hits']}")
print(f"   Misses: {stats['misses']}")
print(f"   Hit Rate: {hit_rate:.1f}%")

print("\n💡 Real-world hit rates:")
print("   • 80%+ is good for user data")
print("   • 95%+ for static content")
print("   • >99% for CDN-cached assets")

## 🧪 Quick Quiz

1. **What's the cache-aside pattern?**

2. **When would you use a short TTL vs long TTL?**

3. **What's write-behind invalidation?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Cache-aside pattern:")
print("   - Check cache first")
print("   - On miss: query DB, store in cache")
print("   - Application controls caching logic")
print()
print("2. TTL choices:")
print("   Short (seconds): Frequently changing data")
print("   Long (hours): Rarely changing data")
print("   Based on staleness tolerance!")
print()
print("3. Write-behind invalidation:")
print("   - Delete cache entry after DB write")
print("   - Next read will have cache miss")
print("   - Simple but causes one extra DB query")

## 📚 Summary

### Key Takeaways

1. **Cache-aside is the default pattern** - check cache, fallback to DB
2. **TTL based on staleness tolerance** - how stale can data be?
3. **Invalidate on writes** - delete or update cache entries
4. **Hit rate matters** - 80%+ is good, 95%+ is great
5. **Popular data stays cached** - Zipf's law helps us!

### Next Up

In **Notebook 6**, we'll cover advanced cache patterns:
- Cache stampede prevention
- Hot key problem
- Cache versioning